# Query 10: Compare CSV vs Parquet Read Performance
**Type:** Optimization – File Format Comparison + Caching + Partitioning  
**Problem Statement:** Quantify the performance difference between reading/querying data in CSV vs Parquet format, and evaluate the impact of caching and partitioning strategies on query execution time.

In [1]:
import time
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName('Q10_FormatOptimization') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

26/04/25 18:34:00 WARN Utils: Your hostname, mariam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 18:34:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 18:34:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/25 18:34:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/25 18:34:02 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/25 18:34:02 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/25 18:34:02 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


Spark version: 3.5.1


## Step 1: Convert CSV to Parquet

In [4]:
import os

csv_path            = '../data/yellow_tripdata_2015-01.csv'
parquet_path        = '../data/yellow_taxi.parquet'
parquet_partitioned = '../data/yellow_taxi_partitioned.parquet'

# Load CSV with sample to avoid disk issues
df_csv = spark.read.option('header', 'true').option('inferSchema', 'true').csv(csv_path) \
              .sample(fraction=0.2, seed=42)
df_csv = df_csv.withColumn('trip_month', F.month('tpep_pickup_datetime'))

# Save as plain Parquet (only if not already created)
if not os.path.exists(parquet_path):
    print('Converting CSV -> Parquet ...')
    df_csv.write.mode('overwrite').parquet(parquet_path)
    print('Done.')
else:
    print('Parquet already exists, skipping.')

# Save as partitioned Parquet by month (only if not already created)
if not os.path.exists(parquet_partitioned):
    print('Converting CSV -> Partitioned Parquet (by month) ...')
    df_csv.write.mode('overwrite').partitionBy('trip_month').parquet(parquet_partitioned)
    print('Done.')
else:
    print('Partitioned Parquet already exists, skipping.')

# Check file sizes
def dir_size_mb(path):
    total = 0
    for dirpath, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(dirpath, f))
    return round(total / (1024*1024), 1)

csv_size     = round(os.path.getsize(csv_path) / (1024*1024), 1) if os.path.exists(csv_path) else '?'
parquet_size = dir_size_mb(parquet_path) if os.path.exists(parquet_path) else '?'
print(f'CSV size:     {csv_size} MB')
print(f'Parquet size: {parquet_size} MB')

Converting CSV -> Parquet ...


Done.
Converting CSV -> Partitioned Parquet (by month) ...


Done.
CSV size:     1680.0 MB
Parquet size: 67.3 MB


## Benchmark Query: Total revenue by payment type (same query, different formats)

In [5]:
def run_aggregation(df, label):
    """Run a standard aggregation and return (result, time)"""
    start = time.time()
    result = (
        df.filter(F.col('fare_amount').isNotNull())
          .groupBy('payment_type')
          .agg(
              F.count('*').alias('trip_count'),
              F.round(F.avg('fare_amount'), 2).alias('avg_fare'),
              F.round(F.sum('total_amount'), 2).alias('total_revenue')
          )
          .orderBy('payment_type')
    )
    result.show()
    elapsed = time.time() - start
    print(f'[{label}] Time: {elapsed:.2f}s')
    return elapsed

In [6]:
# Test 1: CSV (no cache)
print('\n--- TEST 1: CSV (no cache) ---')
df_test_csv = spark.read.csv(csv_path, header=True, inferSchema=True)
t_csv = run_aggregation(df_test_csv, 'CSV')


--- TEST 1: CSV (no cache) ---


+------------+----------+--------+-------------+
|payment_type|trip_count|avg_fare|total_revenue|
+------------+----------+--------+-------------+
|         0.5|         1|    1.76|         NULL|
|         1.0|   6899382|    12.5|1.170542868E8|
|         2.0|   4214289|   10.95|5.147474949E7|
|         3.0|     33807|    10.4|    401063.58|
|         4.0|     10397|    9.76|    112528.06|
|         5.0|         2|     3.0|          6.8|
+------------+----------+--------+-------------+

[CSV] Time: 9.59s


In [7]:
# Test 2: Parquet (no cache)
print('\n--- TEST 2: Parquet (no cache) ---')
df_test_parquet = spark.read.parquet(parquet_path)
t_parquet = run_aggregation(df_test_parquet, 'Parquet')


--- TEST 2: Parquet (no cache) ---


+------------+----------+--------+-------------+
|payment_type|trip_count|avg_fare|total_revenue|
+------------+----------+--------+-------------+
|         1.0|   1381058|   12.49|2.263381911E7|
|         2.0|    844051|   10.93|1.029665685E7|
|         3.0|      6665|   10.65|     80395.77|
|         4.0|      2052|    10.3|     23170.61|
+------------+----------+--------+-------------+

[Parquet] Time: 1.21s


In [8]:
# Test 3: Parquet + Cache (run twice, measure second run)
print('\n--- TEST 3: Parquet + Cache ---')
df_cached = spark.read.parquet(parquet_path).cache()
df_cached.count()  # trigger caching
t_cached = run_aggregation(df_cached, 'Parquet+Cache')
df_cached.unpersist()


--- TEST 3: Parquet + Cache ---


+------------+----------+--------+-------------+
|payment_type|trip_count|avg_fare|total_revenue|
+------------+----------+--------+-------------+
|         1.0|   1381058|   12.49|2.263381911E7|
|         2.0|    844051|   10.93|1.029665685E7|
|         3.0|      6665|   10.65|     80395.77|
|         4.0|      2052|    10.3|     23170.61|
+------------+----------+--------+-------------+

[Parquet+Cache] Time: 1.04s


DataFrame[VendorID: int, tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: string, passenger_count: double, trip_distance: double, pickup_longitude: string, pickup_latitude: double, RateCodeID: double, store_and_fwd_flag: string, dropoff_longitude: double, dropoff_latitude: double, payment_type: double, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, trip_month: int]

In [9]:
# Test 4: Partitioned Parquet (month=1 only — partition pruning)
print('\n--- TEST 4: Partitioned Parquet (January only — partition pruning) ---')
df_partitioned = spark.read.parquet(parquet_partitioned).filter(F.col('trip_month') == 1)
df_partitioned.explain(True)  # Show partition pruning in physical plan
t_partitioned = run_aggregation(df_partitioned, 'Partitioned+Pruning')


--- TEST 4: Partitioned Parquet (January only — partition pruning) ---
== Parsed Logical Plan ==
'Filter ('trip_month = 1)
+- Relation [VendorID#1606,tpep_pickup_datetime#1607,tpep_dropoff_datetime#1608,passenger_count#1609,trip_distance#1610,pickup_longitude#1611,pickup_latitude#1612,RateCodeID#1613,store_and_fwd_flag#1614,dropoff_longitude#1615,dropoff_latitude#1616,payment_type#1617,fare_amount#1618,extra#1619,mta_tax#1620,tip_amount#1621,tolls_amount#1622,improvement_surcharge#1623,total_amount#1624,trip_month#1625] parquet

== Analyzed Logical Plan ==
VendorID: int, tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: string, passenger_count: double, trip_distance: double, pickup_longitude: string, pickup_latitude: double, RateCodeID: double, store_and_fwd_flag: string, dropoff_longitude: double, dropoff_latitude: double, payment_type: double, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amo

## RDD Implementation (CSV only)

In [10]:
start = time.time()

rdd = spark.read.csv(csv_path, header=True, inferSchema=True).rdd

result_rdd = (
    rdd
    .filter(lambda r: r['fare_amount'] is not None and r['payment_type'] is not None)
    .map(lambda r: (r['payment_type'], (float(r['fare_amount']), float(r['total_amount'] or 0), 1)))
    .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1], a[2]+b[2]))
    .mapValues(lambda x: (round(x[0]/x[2], 2), round(x[1], 2), x[2]))
    .sortByKey()
    .collect()
)

t_rdd = time.time() - start
print(f'RDD (CSV) | Time: {t_rdd:.2f}s')
for pt, (avg_f, total_r, cnt) in result_rdd:
    print(f'  payment_type={pt}  avg_fare=${avg_f}  total_revenue=${total_r:,.2f}  trips={cnt:,}')

RDD (CSV) | Time: 101.86s
  payment_type=0.5  avg_fare=$1.76  total_revenue=$0.00  trips=1
  payment_type=1.0  avg_fare=$12.5  total_revenue=$117,054,286.80  trips=6,899,382
  payment_type=2.0  avg_fare=$10.95  total_revenue=$51,474,749.49  trips=4,214,289
  payment_type=3.0  avg_fare=$10.4  total_revenue=$401,063.58  trips=33,807
  payment_type=4.0  avg_fare=$9.76  total_revenue=$112,528.06  trips=10,397
  payment_type=5.0  avg_fare=$3.0  total_revenue=$6.80  trips=2


## Final Performance Summary

In [14]:
print('='*70)
print('QUERY 10 - FORMAT & OPTIMIZATION BENCHMARK')
print('='*70)
header = f'{"Strategy":<35} {"Time":>10} {"vs CSV":>10}'
print(header)
print('-'*70)
results = [
    ('RDD (CSV)', t_rdd),
    ('DataFrame - CSV', t_csv),
    ('DataFrame - Parquet', t_parquet),
    ('DataFrame - Parquet + Cache', t_cached),
    ('DataFrame - Partitioned Parquet (pruned)', t_partitioned),
]
for label, t in results:
    speedup = f'{t_csv/t:.1f}x faster' if t < t_csv else 'baseline'
    print(f'{label:<35} {t:>9.2f}s {speedup:>10}')
print('='*70)
print()
print('KEY INSIGHTS:')
print('1. Parquet reads 3-5x faster than CSV due to columnar storage.')
print('2. Caching eliminates re-read cost for repeated queries.')
print('3. Partition pruning skips irrelevant files entirely.')
print('4. RDD is slowest — no columnar optimization or Catalyst.')
print('5. Parquet uses column-level statistics for predicate pushdown.')

QUERY 10 - FORMAT & OPTIMIZATION BENCHMARK
Strategy                                  Time     vs CSV
----------------------------------------------------------------------
RDD (CSV)                              101.86s   baseline
DataFrame - CSV                          9.59s   baseline
DataFrame - Parquet                      1.21s 7.9x faster
DataFrame - Parquet + Cache              1.04s 9.2x faster
DataFrame - Partitioned Parquet (pruned)      0.83s 11.5x faster

KEY INSIGHTS:
1. Parquet reads 3-5x faster than CSV due to columnar storage.
2. Caching eliminates re-read cost for repeated queries.
3. Partition pruning skips irrelevant files entirely.
4. RDD is slowest — no columnar optimization or Catalyst.
5. Parquet uses column-level statistics for predicate pushdown.
